# Calgary development permits: Before-period exploratory review

Inspect source quality and classification evidence for the full downloaded **Before** snapshot. Reusable behavior lives in `dp_activity`; this notebook calls those modules and displays their results.

**Run cells from top to bottom** with the project's Python interpreter. The pinned snapshot contains 16,637 records downloaded for August 6, 2022–August 5, 2024. This notebook works offline and makes no download requests.

Counts describe the loaded development-permit records, not completed housing units. This single-period review does not establish rezoning effects or a before/during comparison.

Questions:
- Which fields are missing, duplicated, or invalid?
- What changes does cleaning make while preserving source evidence?
- Which rules win, and which records need manual review?
- Which overlapping matches are expected fallbacks versus contradictory evidence?

In [1]:
from pathlib import Path
from dataclasses import asdict
import hashlib
import json
import sys

import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Locate the repository from either its root or the notebooks directory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file() and (path / "src/dp_activity").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the Data110_DP_Activity repository.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from dp_activity.config import load_config
from dp_activity.cli import _default_column_map, _default_classification_field_map
from dp_activity.repositories.raw_data_repository import RawDataRepository
from dp_activity.profiling.data_profiler import DataProfiler
from dp_activity.cleaning.permit_cleaner import PermitCleaner
from dp_activity.classification.rule_loader import RuleLoader
from dp_activity.classification.classifier import PermitClassifier

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_colwidth", 100)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__}")

from dp_activity.validation.schema_validator import SchemaValidator
from dp_activity.validation.data_quality_validator import DataQualityValidator
from dp_activity.validation.classification_validator import ClassificationValidator

Python 3.13.14 | pandas 2.2.3


## Select settings and the frozen Before snapshot

Source identity, study dates, paths, and classification options come from `config/settings.yaml`. The explicit filename pins this review to the downloaded Parquet snapshot so a newer download does not silently change the results.

Set `SNAPSHOT_FILENAME` to another frozen snapshot under the configured raw directory when deliberately changing inputs. The notebook verifies its checksum, row count, dataset identity, query dates, and every application date against `study_periods.before`. Missing files or mismatched settings stop execution; no sample is downloaded as a fallback. `REVIEW_LIMIT` limits displayed examples only, not analyzed records.

In [9]:
SETTINGS_PATH = ROOT / "config/settings.yaml"
STUDY_PERIOD_KEY = "before"
SNAPSHOT_FILENAME = "development_permits_raw_Before_2022-08-06_2024-08-05_20260922_001829.parquet"
REVIEW_LIMIT = 10

config = load_config(SETTINGS_PATH)
study_period = config.raw["study_periods"][STUDY_PERIOD_KEY]
snapshot_path = config.raw_data_dir / SNAPSHOT_FILENAME
period_start = pd.Timestamp(study_period["start"])
period_end_exclusive = pd.Timestamp(study_period["end"]) + pd.Timedelta(1, unit="D")
primary_date_field = config.raw["analysis"]["primary_date_field"]
source_date_field = next(
    (source for source, target in _default_column_map().items() if target == primary_date_field),
    None,
)
if source_date_field is None:
    raise ValueError("The configured primary date field has no source-column mapping.")
expected_where = (
    f"({source_date_field} >= '{period_start.date().isoformat()}T00:00:00' AND "
    f"{source_date_field} < '{period_end_exclusive.date().isoformat()}T00:00:00')"
)

display(pd.DataFrame({
    "setting": ["dataset_id", "study_period", "start_inclusive", "end_inclusive",
                "primary_date_field", "rule_file", "raw_formats"],
    "value": [config.raw["data_source"]["dataset_id"], study_period["label"],
              study_period["start"], study_period["end"], primary_date_field,
              str(config.classification_rules_path.relative_to(ROOT)),
              ", ".join(config.raw_snapshot_formats)],
}))

,setting,value
0,dataset_id,6933-unw5
1,study_period,Before
2,start_inclusive,2022-08-06
3,end_inclusive,2024-08-05
4,primary_date_field,applied_date
5,rule_file,config\classification_rules.csv
6,raw_formats,"csv, parquet"


In [3]:
if not snapshot_path.is_file():
    raise FileNotFoundError(
        f"Frozen snapshot not found: {snapshot_path.name}. "
        "Download the selected study period or update SNAPSHOT_FILENAME."
    )
sidecar_path = snapshot_path.with_suffix(snapshot_path.suffix + ".metadata.json")
if not sidecar_path.is_file():
    raise FileNotFoundError("The selected snapshot requires its metadata sidecar.")
print("Using the complete frozen Before snapshot; no network request was made.")
print(f"Selected snapshot: {snapshot_path.relative_to(ROOT)}")

Using the complete frozen Before snapshot; no network request was made.
Selected snapshot: data\raw\development_permits_raw_Before_2022-08-06_2024-08-05_20260922_001829.parquet


## Load and verify provenance

Verify the saved checksum and row count, dataset identity, and download query. Check all application dates against the configured study interval, including its final day. Configuration and rule hashes identify the definitions used for this review; credentials are never displayed.

These checks establish consistency with the saved download, not independent completeness of the City's source. “Raw” means the repository-loaded table before cleaning; consult the snapshot for exact source serialization. CSV loading can infer types and missing-value markers.

In [4]:
repository = RawDataRepository(config.raw_data_dir)
raw = repository.load_snapshot(snapshot_path)
raw_before = raw.copy(deep=True)
digest = hashlib.sha256()
with snapshot_path.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        digest.update(chunk)
snapshot_sha256 = digest.hexdigest()
sidecar_path = snapshot_path.with_suffix(snapshot_path.suffix + ".metadata.json")
metadata = json.loads(sidecar_path.read_text(encoding="utf-8"))
if metadata.get("sha256") != snapshot_sha256:
    raise ValueError("Snapshot checksum differs from its metadata sidecar.")
if metadata.get("row_count") != len(raw):
    raise ValueError("Loaded row count differs from snapshot metadata.")

if metadata.get("dataset_id") != config.raw["data_source"]["dataset_id"]:
    raise ValueError("Snapshot dataset does not match the configured source.")
query = metadata.get("query", {})
if isinstance(query, str):
    query = json.loads(query)
if query.get("where") != expected_where:
    raise ValueError("Snapshot query does not match the configured study period.")
application_dates = pd.to_datetime(raw[source_date_field], errors="raise")
if raw.empty or not (
    application_dates.notna()
    & application_dates.ge(period_start)
    & application_dates.lt(period_end_exclusive)
).all():
    raise ValueError("Snapshot contains missing dates or dates outside the selected study period.")

display(pd.DataFrame([{
    "study_period": study_period["label"],
    "first_application": application_dates.min(),
    "last_application": application_dates.max(),
    "period_verified": True,
    "rows": len(raw),
    "columns": len(raw.columns),
    "snapshot_sha256": snapshot_sha256,
    "checksum_verified": bool(metadata.get("sha256")),
    "retrieved_at_utc": metadata.get("retrieved_at_utc", "unknown"),
    "selection": "Full Before-period download; no sample limit",
    "settings_sha256": hashlib.sha256(SETTINGS_PATH.read_bytes()).hexdigest(),
    "rules_sha256": hashlib.sha256(config.classification_rules_path.read_bytes()).hexdigest(),
}]))
preview_columns = [name for name in ("permitnum", "applieddate", "category", "proposedusedescription", "communityname") if name in raw]
display(raw[preview_columns].head(REVIEW_LIMIT))

,study_period,first_application,last_application,period_verified,rows,columns,snapshot_sha256,checksum_verified,retrieved_at_utc,selection,settings_sha256,rules_sha256
0,Before,2022-08-06,2024-08-05,True,16637,34,649e5e5d95b80003543b03facc372a5beeaea15746687c542f89b52798eded32,True,2026-09-22 00:18:29.189616+00:00,Full Before-period download; no sample limit,b8f93c31688081c2d0724d029b9be27940d35afc09e7f31f447f4c7590913a8d,2dc55f173351e9caac7e41d0083b464436234b31a1b55b9c5418524ddeb1636c


,permitnum,applieddate,category,proposedusedescription,communityname
0,DP2022-05518,2022-08-07T00:00:00.000,Residential - Secondary Suite,SECONDARY SUITE,CITYSCAPE
1,DP2022-05515,2022-08-07T00:00:00.000,Residential - Secondary Suite,SECONDARY SUITE,SILVER SPRINGS
2,DP2022-05516,2022-08-07T00:00:00.000,Residential - Secondary Suite,BACKYARD SUITE,WHITEHORN
3,DP2022-05517,2022-08-07T00:00:00.000,Residential - Secondary Suite,SECONDARY SUITE,EVANSTON
4,DP2022-05513,2022-08-06T00:00:00.000,Home Occupation Class 2,HOME OCCUPATION - CLASS 2,SANDSTONE VALLEY
5,DP2022-05512,2022-08-06T00:00:00.000,Residential - Secondary Suite,SECONDARY SUITE,SKYVIEW RANCH
6,DP2022-05514,2022-08-06T00:00:00.000,Signs - Permitted Use,SIGN - CLASS B,BELTLINE
7,DP2022-05549,2022-08-09T00:00:00.000,Home Occupation Class 2,HOME OCCUPATION - CLASS 2,TEMPLE
8,DP2022-05550,2022-08-09T00:00:00.000,Change of Use - Permitted Use,RETAIL AND CONSUMER SERVICE,SHAGANAPPI
9,DP2022-05551,2022-08-09T00:00:00.000,Relaxation - Existing - Compliance Follow-Up,DECK,GARRISON WOODS


## Profile the source

Percentages use all loaded rows as the denominator. The profiler counts pandas nulls as missing; blank strings remain evidence until cleaning. Duplicate examples are capped at 20 per identifier field, but counts include all rows. Date bounds in profiler reports are normalized to UTC for comparison and are not policy-period assignments.

In [5]:
profiler = DataProfiler()
source_categories = [
    name for name in ("category", "proposedusedescription", "landusedistrict", "statuscurrent")
    if name in raw
]
raw_profile = profiler.profile(raw, source_categories)
display(raw_profile["summary"])
display(raw_profile["dtypes"])
display(raw_profile["missingness"].sort_values("missing_percentage", ascending=False).head(15))
display(raw_profile["permit_uniqueness"])
display(raw_profile["duplicate_examples"].head(REVIEW_LIMIT))
display(raw_profile["date_ranges"])
for name in source_categories:
    print(f"Source distribution: {name}")
    display(raw_profile[name].head(REVIEW_LIMIT))

,row_count,column_count
0,16637,34


,column,dtype
0,point,object
1,permitnum,object
2,address,object
3,applicant,object
4,category,object
...,...,...
29,canceledrefuseddate,object
30,sdabnumber,object
31,sdabhearingdate,object
32,sdabdecision,object


,column,missing_count,missing_percentage
31,sdabhearingdate,16485,99.086374
32,sdabdecision,16485,99.086374
30,sdabnumber,16484,99.080363
33,sdabdecisiondate,16482,99.068342
29,canceledrefuseddate,13365,80.332993
3,applicant,7945,47.755004
14,releasedate,3895,23.411673
15,mustcommencedate,3225,19.384504
13,decisiondate,3125,18.783435
17,decisionby,3075,18.482900


,column,non_missing_count,unique_count,duplicate_row_count,duplicate_excess_count
0,permitnum,16637,16637,0,0


,column,row_position,permit_number


...

,value,count,percentage
0,SECONDARY SUITE,3858,23.189277
1,SINGLE DETACHED DWELLING,1368,8.222636
2,HOME OCCUPATION - CLASS 2,1223,7.351085
3,SIGN - CLASS B,1149,6.906293
4,DECK,918,5.517822
5,ACCESSORY RESIDENTIAL BUILDING,771,4.634249
6,RETAIL AND CONSUMER SERVICE,342,2.055659
7,GENERAL INDUSTRIAL - LIGHT,290,1.743103
8,MULTI-RESIDENTIAL DEVELOPMENT,275,1.652942
9,CHILD CARE SERVICE,267,1.604857


Source distribution: landusedistrict


,value,count,percentage
0,R-C1,3372,20.268077
1,R-C2,1812,10.891387
2,R-1N,1767,10.620905
3,DC,1587,9.538979
4,I-G,929,5.583939
5,R-CG,871,5.235319
6,R-G,790,4.748452
7,R-1,645,3.876901
8,I-C,299,1.797199
9,R-C1N,298,1.791188


Source distribution: statuscurrent


,value,count,percentage
0,Released,12772,76.768648
1,Cancelled,3017,18.134279
2,In Advertising,255,1.532728
3,Pending Release,173,1.039851
4,Lapsed,113,0.679209
5,Refused,101,0.607081
6,Hold,49,0.294524
7,Inactive,29,0.174310
8,Cancelled - Pending Refund,28,0.168300
9,New,24,0.144257


## Clean and compare

Use the same field mappings as the CLI. The cleaner preserves source values in `raw_*` columns, retains duplicate records and the index, and produces invalid-date/coordinate flags. It preserves source wall-clock dates rather than shifting them across time zones.

Acceptance checks: row count, index, and source table remain unchanged. Added raw evidence and invalid flags explain why the cleaned table has more columns. Validation modules are still pending, so these checks are exploratory safeguards rather than complete data validation.

In [6]:
column_map = _default_column_map()
analysis_settings = config.raw["analysis"]
date_columns = list(dict.fromkeys([
    analysis_settings["primary_date_field"],
    analysis_settings["processing_time"]["end_field"],
]))
cleaner = PermitCleaner(column_map, date_columns)
cleaned = cleaner.clean(raw)
assert_frame_equal(raw, raw_before)
assert len(cleaned) == len(raw)
assert cleaned.index.equals(raw.index)

clean_categories = [column_map.get(name, name) for name in source_categories]
clean_profile = profiler.profile(cleaned, clean_categories)
before_missing = raw_profile["missingness"].copy()
before_missing["column"] = before_missing["column"].replace(column_map)
comparison = before_missing[["column", "missing_count"]].merge(
    clean_profile["missingness"][["column", "missing_count"]],
    on="column", how="left", suffixes=("_before", "_after"),
)
comparison["change"] = comparison["missing_count_after"] - comparison["missing_count_before"]
display(comparison.sort_values("change", ascending=False).head(15))
invalid_columns = [name for name in cleaned.columns if name.endswith("_invalid")]
display(cleaned[invalid_columns].sum().rename("flagged_rows").to_frame())
print(f"Preserved {len(cleaned)} records and the original row index.")

,column,missing_count_before,missing_count_after,change
0,point,25,25,0
1,permit_number,0,0,0
2,address,1,1,0
3,applicant,7945,7945,0
4,category,7,7,0
5,description,0,0,0
6,proposed_use_code,0,0,0
7,proposed_use_description,0,0,0
8,permitteddiscretionary,0,0,0
9,land_use_district,0,0,0


,flagged_rows
applied_date_invalid,0
decision_date_invalid,0
latitude_invalid,0
longitude_invalid,0


Preserved 16637 records and the original row index.


## Classify with the configured rule file

The first enabled rule in priority/ID order wins. Later matches are retained as potential overlaps. Broad fallback and catch-all rules intentionally overlap specific rules; overlap counts alone do not establish a classification error.

Unmatched records and winners labelled provisional, fallback, or review require manual review. A rule's validation label is not independent proof that the classification is correct.

In [7]:
rules = RuleLoader().load(config.classification_rules_path)
classification_settings = analysis_settings["classification"]
classifier = PermitClassifier(
    rules,
    _default_classification_field_map(),
    unmatched_action=classification_settings["unmatched_action"],
    case_sensitive=classification_settings["case_sensitive"],
)
cleaned_before = cleaned.copy(deep=True)
classified = classifier.classify(cleaned)
coverage = classifier.coverage_report(classified)
assert_frame_equal(cleaned, cleaned_before)
assert len(classified) == len(raw)
assert int(coverage["record_count"].sum()) == len(raw)

display(pd.DataFrame([{
    "enabled_rules": len(rules),
    "records": len(classified),
    "matched": int(classified["ClassificationRule"].notna().sum()),
    "unmatched": int(classified["ClassificationRule"].isna().sum()),
    "needs_review": int(classified["ClassificationNeedsReview"].sum()),
    "overlapping_matches": int(classified["ClassificationConflictCount"].gt(0).sum()),
    "included_residential": int(classified["IncludeResidential"].sum()),
}]))
display(coverage.sort_values("record_count", ascending=False).head(15))
display(classified.groupby(
    ["ResidentialType", "IncludeResidential", "ValidationStatus"], dropna=False
).size().rename("records").reset_index().sort_values("records", ascending=False).head(15))

,enabled_rules,records,matched,unmatched,needs_review,overlapping_matches,included_residential
0,34,16637,16632,5,1101,15914,7940


,ClassificationRule,ValidationStatus,ClassificationNeedsReview,record_count,matched_count,unmatched_count,conflict_count,additional_match_count,record_percentage
0,HF-016,validated_2026_sample,False,3831,3831,0,3831,15345,23.026988
4,HF-905,validated_2026_sample,False,2529,2529,0,2529,2529,15.201058
12,HF-010,validated_2026_sample,False,1856,1856,0,1856,6954,11.155857
3,HF-900,validated_2026_sample,False,1845,1845,0,1845,1845,11.089740
2,HF-901,validated_2026_sample,False,1224,1224,0,1224,1224,7.357096
5,HF-906,validated_2026_sample,False,1103,1103,0,1103,1103,6.629801
10,HF-018,validated_2026_sample,False,814,814,0,814,1641,4.892709
8,HF-999,review,True,718,718,0,0,0,4.315682
16,HF-005,validated_2026_sample,False,482,482,0,482,2643,2.897157
17,HF-012,validated_2026_sample,False,418,418,0,418,1753,2.512472


,ResidentialType,IncludeResidential,ValidationStatus,records
6,Non-Residential,False,validated_2026_sample,5793
14,Secondary Suite,True,validated_2026_sample,3846
16,Single Detached,True,validated_2026_sample,2313
11,Residential Non-Housing,False,validated_2026_sample,1103
1,Accessory Residential Building,False,validated_2026_sample,814
19,Unclassified,False,review,718
15,Semi-Detached,True,validated_2026_sample,525
4,Multi-Residential,True,validated_2026_sample,447
2,Backyard Suite,True,validated_2026_sample,355
13,Rowhouse,True,validated_2026_sample,323


## Inspect review cases and rule coverage

The examples below use source order and are not a randomized validation sample. Read the original evidence, the winning rule, and later matching rules before proposing changes. A zero winning count may mean a rule is shadowed, or simply absent from this small sample; it does not justify deleting that rule.

In [8]:
review_columns = [name for name in (
    "permit_number", "category", "proposed_use_description", "description",
    "raw_category", "raw_proposed_use_description", "ClassificationRule",
    "ValidationStatus", "ResidentialType", "ClassificationNeedsReview",
    "ClassificationMatchedRules",
) if name in classified]
for label, mask in {
    "Unmatched": classified["ClassificationRule"].isna(),
    "Needs review": classified["ClassificationNeedsReview"],
    "Potential overlaps": classified["ClassificationConflictCount"].gt(0),
}.items():
    print(f"{label}: {int(mask.sum())} records; showing at most {REVIEW_LIMIT}.")
    display(classified.loc[mask, review_columns].head(REVIEW_LIMIT))

winning_counts = classified["ClassificationRule"].value_counts()
all_match_counts = classified["ClassificationMatchedRules"].explode().dropna().value_counts()
rule_coverage = pd.DataFrame([{
    "rule_id": rule.rule_id, "priority": rule.priority, "field": rule.field,
    "match_value": rule.match_value, "validation_status": rule.validation_status,
    "winning_records": int(winning_counts.get(rule.rule_id, 0)),
    "all_matching_records": int(all_match_counts.get(rule.rule_id, 0)),
} for rule in rules])
display(rule_coverage)
display(raw_profile["description_examples"].head(REVIEW_LIMIT))

Unmatched: 5 records; showing at most 10.


,permit_number,category,proposed_use_description,description,raw_category,raw_proposed_use_description,ClassificationRule,ValidationStatus,ResidentialType,ClassificationNeedsReview,ClassificationMatchedRules
433,DP2022-05985,None,VEHICLE SALES - MAJOR,CHANGE OF USE: VEHICLE SALES - MAJOR,None,VEHICLE SALES - MAJOR,<NA>,unmatched,Review,True,()
2534,DP2022-08287,None,RETAIL AND CONSUMER SERVICE,CHANGE OF USE: RETAIL AND CONSUMER SERVICE,None,RETAIL AND CONSUMER SERVICE,<NA>,unmatched,Review,True,()
9474,DP2023-07200,None,BUILDING SUPPLY CENTRE,CHANGE OF USE: BUILDING SUPPLY CENTRE,None,BUILDING SUPPLY CENTRE,<NA>,unmatched,Review,True,()
9981,DP2023-07748,None,HEALTH CARE SERVICE,REVISION: HEALTH CARE SERVICE (CHANGE OF USE TO DP2020-3951),None,HEALTH CARE SERVICE,<NA>,unmatched,Review,True,()
14595,DP2024-03548,None,BED AND BREAKFAST,: BED AND BREAKFAST,None,BED AND BREAKFAST,<NA>,unmatched,Review,True,()


Needs review: 1101 records; showing at most 10.


,permit_number,category,proposed_use_description,description,raw_category,raw_proposed_use_description,ClassificationRule,ValidationStatus,ResidentialType,ClassificationNeedsReview,ClassificationMatchedRules
12,DP2022-05555,Commercial - Other Areas,CAR WASH - MULTI-VEHICLE,NEW: CAR WASH - MULTI-VEHICLE,Commercial - Other Areas,CAR WASH - MULTI-VEHICLE,HF-902,fallback,Non-Residential,True,"(HF-902, HF-999)"
14,DP2022-05519,Temporary Structure,POST-SECONDARY LEARNING INSTITUTION,TEMPORARY USE: POST-SECONDARY LEARNING INSTITUTION (RECREATIONAL FACILITY BUILDING - TEN YEARS),Temporary Structure,POST-SECONDARY LEARNING INSTITUTION,HF-999,review,Unclassified,True,"(HF-999,)"
40,DP2022-05557,Commercial - Other Areas,CHILD CARE SERVICE; LIQUOR STORE; RETAIL AND CONSUMER SERVICE,"NEW: LIQUOR STORE, CHILD CARE SERVICE, RETAIL AND CONSUMER SERVICE (6 PHASES, 5 BUILDINGS)",Commercial - Other Areas,CHILD CARE SERVICE; LIQUOR STORE; RETAIL AND CONSUMER SERVICE,HF-902,fallback,Non-Residential,True,"(HF-902, HF-999)"
48,DP2022-05564,Renovations - Non-Residential,DECK,REVISION: MULTI-RESIDENTIAL (DECK),Renovations - Non-Residential,DECK,HF-999,review,Unclassified,True,"(HF-999,)"
82,DP2022-05602,Renovations - Non-Residential,GENERAL INDUSTRIAL - LIGHT,EXTERIOR RENOVATIONS: GENERAL INDUSTRIAL - LIGHT (REFURBISH BUILDING FAÇADE),Renovations - Non-Residential,GENERAL INDUSTRIAL - LIGHT,HF-999,review,Unclassified,True,"(HF-999,)"
89,DP2022-05613,Commercial - Other Areas,RETAIL AND CONSUMER SERVICE; VETERINARY CLINIC,"NEW: VETERINARY CLINIC, RETAIL AND CONSUMER SERVICE (1 BUILDING)",Commercial - Other Areas,RETAIL AND CONSUMER SERVICE; VETERINARY CLINIC,HF-902,fallback,Non-Residential,True,"(HF-902, HF-999)"
91,DP2022-05614,Temporary Structure,GENERAL INDUSTRIAL - LIGHT,TEMPORARY USE: GENERAL INDUSTRIAL - LIGHT (STORAGE TENT) - 5 YEARS,Temporary Structure,GENERAL INDUSTRIAL - LIGHT,HF-999,review,Unclassified,True,"(HF-999,)"
121,DP2022-05630,Development Design Guidelines,TEMPORARY RESIDENTIAL SALES CENTRE,TEMPORARY USE: TEMPORARY RESIDENTIAL SALES CENTRE (3 UNITS) - 2 YEARS,Development Design Guidelines,TEMPORARY RESIDENTIAL SALES CENTRE,HF-999,review,Unclassified,True,"(HF-999,)"
133,DP2022-05659,Temporary Structure,TEMPORARY RESIDENTIAL SALES CENTRE,TEMPORARY USE: TEMPORARY RESIDENTIAL SALES CENTRE (4 YEARS),Temporary Structure,TEMPORARY RESIDENTIAL SALES CENTRE,HF-999,review,Unclassified,True,"(HF-999,)"
154,DP2022-05680,Renovations - Non-Residential,"BREWERY, WINERY AND DISTILLERY; OUTDOOR CAFE; RESTAURANT: LICENSED","EXTERIOR RENOVATIONS: BREWERY, WINERY AND DISTILLERY, RESTAURANT: LICENSED (NEW GARAGE DOOR); RE...",Renovations - Non-Residential,"BREWERY, WINERY AND DISTILLERY; OUTDOOR CAFE; RESTAURANT: LICENSED",HF-999,review,Unclassified,True,"(HF-999,)"


Potential overlaps: 15914 records; showing at most 10.


,permit_number,category,proposed_use_description,description,raw_category,raw_proposed_use_description,ClassificationRule,ValidationStatus,ResidentialType,ClassificationNeedsReview,ClassificationMatchedRules
0,DP2022-05518,Residential - Secondary Suite,SECONDARY SUITE,NEW: SECONDARY SUITE (BASEMENT) - PARKING STALL SIZE,Residential - Secondary Suite,SECONDARY SUITE,HF-016,validated_2026_sample,Secondary Suite,False,"(HF-016, HF-017, HF-102, HF-106, HF-999)"
1,DP2022-05515,Residential - Secondary Suite,SECONDARY SUITE,NEW: SECONDARY SUITE (BASEMENT),Residential - Secondary Suite,SECONDARY SUITE,HF-016,validated_2026_sample,Secondary Suite,False,"(HF-016, HF-017, HF-102, HF-106, HF-999)"
2,DP2022-05516,Residential - Secondary Suite,BACKYARD SUITE,NEW: BACKYARD SUITE (CONVERTING EXISTING GARAGE),Residential - Secondary Suite,BACKYARD SUITE,HF-014,validated_2026_sample,Backyard Suite,False,"(HF-014, HF-015, HF-102, HF-106, HF-999)"
3,DP2022-05517,Residential - Secondary Suite,SECONDARY SUITE,NEW: SECONDARY SUITE (BASEMENT),Residential - Secondary Suite,SECONDARY SUITE,HF-016,validated_2026_sample,Secondary Suite,False,"(HF-016, HF-017, HF-102, HF-106, HF-999)"
4,DP2022-05513,Home Occupation Class 2,HOME OCCUPATION - CLASS 2,TEMPORARY USE: HOME OCCUPATION - CLASS 2 (MOTOR VEHICLE DEALER - 18 MONTHS),Home Occupation Class 2,HOME OCCUPATION - CLASS 2,HF-901,validated_2026_sample,Non-Residential,False,"(HF-901, HF-999)"
5,DP2022-05512,Residential - Secondary Suite,SECONDARY SUITE,NEW: SECONDARY SUITE (BASEMENT),Residential - Secondary Suite,SECONDARY SUITE,HF-016,validated_2026_sample,Secondary Suite,False,"(HF-016, HF-017, HF-102, HF-106, HF-999)"
6,DP2022-05514,Signs - Permitted Use,SIGN - CLASS B,NEW: SIGN - CLASS B (FASCIA SIGN),Signs - Permitted Use,SIGN - CLASS B,HF-900,validated_2026_sample,Non-Residential,False,"(HF-900, HF-999)"
7,DP2022-05549,Home Occupation Class 2,HOME OCCUPATION - CLASS 2,TEMPORARY USE: HOME OCCUPATION - CLASS 2 (AESTHETICS - 18 MONTHS),Home Occupation Class 2,HOME OCCUPATION - CLASS 2,HF-901,validated_2026_sample,Non-Residential,False,"(HF-901, HF-999)"
8,DP2022-05550,Change of Use - Permitted Use,RETAIL AND CONSUMER SERVICE,REVISION: RETAIL AND CONSUMER SERVICE (CHANGE OF USE TO DP2016-4510),Change of Use - Permitted Use,RETAIL AND CONSUMER SERVICE,HF-905,validated_2026_sample,Non-Residential,False,"(HF-905, HF-999)"
9,DP2022-05551,Relaxation - Existing - Compliance Follow-Up,DECK,RELAXATION: PRIVACY WALL (EXISTING) - HEIGHT,Relaxation - Existing - Compliance Follow-Up,DECK,HF-906,validated_2026_sample,Residential Non-Housing,False,"(HF-906, HF-999)"


,rule_id,priority,field,match_value,validation_status,winning_records,all_matching_records
0,HF-001,10,proposedusedescription,ROWHOUSE BUILDING,validated_2026_sample,282,282
1,HF-002,11,description,ROWHOUSE,validated_2026_sample,41,315
2,HF-003,20,proposedusedescription,TOWNHOUSE,provisional,54,60
3,HF-004,21,description,TOWNHOUSE,validated_2026_sample,7,72
4,HF-005,30,proposedusedescription,SEMI-DETACHED DWELLING,validated_2026_sample,482,494
...,...,...,...,...,...,...,...
29,HF-903,903,category,Industrial,validated_2026_sample,137,137
30,HF-904,904,category,Mixed Use,validated_2026_sample,58,84
31,HF-905,905,category,Change of Use,validated_2026_sample,2529,2588
32,HF-906,906,category,Relaxation,validated_2026_sample,1103,2892


,column,row_position,description,character_count
0,description,2913,REVISION: ASSISTED LIVING (PARKING & MECHANICAL EQUIPMENT) - 60(1)(C) ALL ELECTRICAL AND MECHANI...,416
1,description,4157,"REVISION: RETAIL AND CONSUMER SERVICE, CHILD CARE SERVICE (168 CHILDREN, OUTDOOR PLAY AREA) HEAL...",410
2,description,14707,RELAXATION: SINGLE DETACHED DWELLING (EXISTING) - BUILDING SETBACK FROM SIDE & REAR PROPERTY LIN...,390
3,description,9815,CHANGE OF USE: RESTAURANT (2) (WITHIN EXISTING AUTOMOTIVE SERVICE); EXTERIOR RENOVATIONS: AUTOMO...,369
4,description,534,"CHANGE OF USE: BREWERY, WINERY AND DISTILLERY; EXTERIOR RENOVATIONS: BREWERY, WINERY AND DISTILL...",329
5,description,14551,"RELAXATION: SINGLE DETACHED DWELLING (EXISTING) - BUILDING SETBACK FROM SIDE PROPERTY LINE, ACCE...",320
6,description,4935,CHANGE OF USE: SINGLE DETACHED DWELLING; RELAXATION: SINGLE DETACHED DWELLING (EXISTING) - BUILD...,316
7,description,4931,CHANGES TO SITE PLAN: CHANGE OF USE: DWELLING UNITS; EXTERIOR RENOVATIONS: MULTI-RESIDENTIAL DEV...,306
8,description,3476,"RELAXATION: SINGLE DETACHED DWELLING (EXISTING) BUILDING SETBACK FROM SIDE & REAR PROPERTY LINE,...",292
9,description,6607,"NEW: RETAIL AND CONSUMER SERVICE, FINANCIAL INSTITUTION, CHILD CARE SERVICE (208 CHILDREN), REST...",283


## Validate schema, data quality, and classification

Run the reusable validation strategies against the full classified table. Required columns and quality options come from `config/settings.yaml`; classification validation uses the same loaded rules as the classifier. Schema and quality result objects are converted to display tables without changing their meaning.

Additional source and derived columns are informational. Missing prerequisites and inconsistent audit fields are failures; missing values and review cases may be warnings. Findings remain visible for investigation and do not remove records or automatically stop the notebook. Optional size/freshness checks run only when configured; freshness uses an explicit reference date suitable for the study window.

No independent human-labelled audit sample is supplied here, so accuracy remains unknown and no confusion matrix is produced. Coverage and rule status labels do not establish accuracy.

In [ ]:
validation_input = classified.copy(deep=True)
quality_settings = config.raw["quality_checks"]
schema_issues = SchemaValidator().validate(
    classified, required_columns=quality_settings["required_columns"],
)
schema_report = pd.DataFrame(
    [asdict(issue) for issue in schema_issues],
    columns=["severity", "column", "message"],
)
quality_results = DataQualityValidator().validate(classified, settings=quality_settings)
quality_report = pd.DataFrame(
    [asdict(result) for result in quality_results],
    columns=["check_name", "status", "affected_rows", "message"],
)
classification_validation = ClassificationValidator().validate(classified, rules=rules)

assert_frame_equal(classified, validation_input)
assert_frame_equal(cleaned, cleaned_before)
assert_frame_equal(raw, raw_before)
summary = classification_validation.loc[
    classification_validation["report_type"].eq("summary")
].iloc[0]
assert int(summary["record_count"]) == len(classified)
assert int(summary["matched_count"] + summary["unmatched_count"]) == len(classified)

print("Schema findings by severity (additional columns are informational):")
display(schema_report.groupby("severity").size().rename("findings").to_frame())
display(schema_report.loc[schema_report["severity"].eq("error")])
print(f"Additional columns: showing at most {REVIEW_LIMIT}; full results are in schema_report.")
display(schema_report.loc[schema_report["severity"].eq("info")].head(REVIEW_LIMIT))

print("Data-quality checks: affected counts can overlap and must not be summed.")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 160):
    display(quality_report)
print("Optional minimum_row_count enabled:", "minimum_row_count" in quality_settings)
print("Optional freshness enabled:", "max_data_age_days" in quality_settings)

print("Classification validation: coverage is not accuracy.")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 160):
    display(summary.to_frame("value"))
print("All validation strategies preserved input records.")


## Findings and next actions

Use the refreshed full-period tables and validator reports above as evidence. Validation warnings require review; successful execution does not mean every record is complete or correct. No independent human-label accuracy assessment has been performed.

Before accepting a rule change:
1. Inspect unmatched records, review flags, and failed or warning validation checks.
2. Retain representative positive and negative examples, including permit and rule IDs.
3. Distinguish missing evidence, cleaning issues, precedence, and rule wording. Multiple matches alone are not classification errors.
4. Move reusable findings into module tests or the versioned rule CSV.
5. Rerun against the same frozen snapshot and compare outcomes.

**Implemented and executed here:** schema, data-quality, and classification validators.

**Still pending:** policy-period and seasonal features, processing-time features, final analyses, exports, chart generation, an end-to-end smoke test, and Power BI reconciliation. These reports do not establish a before/during comparison.

Raw snapshots stay in the ignored data directory. Notebook outputs can contain public source descriptions; review or clear outputs before sharing.